# CO2Dot calibration run 2 (multi-device, LI-6800)

Protocol 2: steady-state windows on a fixed RH x T x CO2 grid, repeated until stopped.
All logic lives in `co2dot_calib2.py` (scheduler, auto-tune) and `co2dot_calib.py`
(devices, LI-6800 adapter, loader); this notebook only configures and starts it.

**Protocol**
- LED at the driver minimum (4 mA). Gain and atime are auto-tuned once per device at start,
  in ambient air, so the brightest lit channel reads ~30 000 counts (46 % of 65 535). The
  brightest state (0 or 2000 ppm) is then expected in the 50-60 % window; the quick-look cell
  checks that and a `saturation` event is logged if any sample exceeds 95 %.
- Grid: RH 20, 30, 40, 50, 60 % (outer) > T 17, 22, 27 C > CO2 0, 300, 800, 2000, 0 ppm.
  65 % and 80 % at 27 C need a 19.9 / 23.3 C dew point, above the humidifier ceiling (~19.7 C),
  and 80 % at 17 / 22 C sits within 3.5 C of condensation on the exchanger, so 60 % is the top.
- Per CO2 step: setpoint, wait until CO2_s is within tolerance (3 ppm or 1 % for 300/800,
  < 15 ppm for zero, 40 ppm for 2000; no stability flag), then **one 60 s window at 5 s cadence**.
  Nothing is logged outside the windows. Timeout 5 min: the window is recorded anyway, tagged `reached=False`.
- A new (RH, T) cell sets T, RH and the first CO2 together and waits for all three; no extra hold.
- +-30 ppm jitter on 300 and 800 only. Odd passes walk the grid in reverse.
- Ctrl-C (interrupt the kernel) stops cleanly: chamber restored, checkpoint kept; `run.run()` resumes.

**Before starting**
1. Devices in the chamber facing the air stream; LI-6800 leaf thermocouple taped to one film.
2. Fresh CO2 cartridge, soda lime, desiccant and humidifier columns checked.
3. Device identity = USB serial number (chip MAC). Fill `LABELS` below once per device.
4. Auto-tune must report OK on every device (peak near 30 000 counts, no warnings).

In [ ]:
import time
from datetime import datetime
from pathlib import Path
from co2dot_calib import connect_dots, preflight, LicorLive, MockLicor, load_run, settled_points
from co2dot_calib2 import autotune, Protocol2, Run2, mock_dots2

## 1. Devices and spectrometer auto-tune

In [ ]:
LABELS = {                      # device_id (USB serial = MAC) -> bench label; extend as devices are added
    "3C:DC:75:0D:FA:64": "A",
}
LED_MA = 4                      # driver minimum (4 mA, 2 mA steps, 20 mA max)
dots = connect_dots(labels=LABELS)
tune = autotune(dots, led_ma=LED_MA, target_counts=30000)   # sets gain/atime per device; astep stays 999
report = preflight(dots, led_ma=LED_MA, n=3)

## 2. LI-6800

In [ ]:
from li_mqtt import LI6800
li = LI6800.connect()
print("broker", li.broker)
time.sleep(2)
print({k: li.get(k) for k in ("CO2_s", "CO2_r", "H2O_s", "Tchamber", "Txchg", "Tleaf", "RHcham", "Flow", "Press")})
licor = LicorLive(li)

## 3. Protocol

Edit the grid and timings here. `max_dew_c` is the humidifier ceiling: cells whose dew point is above
it are skipped (logged). `passes=0` repeats until the kernel is interrupted.

In [ ]:
P = Protocol2(
    rh_levels=(20, 30, 40, 50, 60),
    t_levels=(17, 22, 27),
    co2_levels=(0, 300, 800, 2000, 0),
    co2_jitter_ppm=30,                   # on 300 and 800 only
    co2_zero_tol_ppm=15, co2_top_tol_ppm=40, co2_tol_ppm=3, co2_timeout_s=300,
    t_tol_c=0.5, t_timeout_s=1500, rh_tol_pct=3, rh_timeout_s=1800,
    sample_period_s=5, sample_window_s=60,
    passes=0,
    max_dew_c=19.5, cooldown_dry=True,
    flow_umol_s=400,
)
print(P.feasibility_table())
est = P.estimate()
print(f"\n{est['cells']} cells, {est['co2_steps']} CO2 steps, {est['skipped']} skipped per pass;  "
      f"~{est['hours_per_pass']} h per pass, mixer active ~{est['mixer_hours_per_pass']} h per pass")

## 4. Dry run without hardware (optional, ~3 min)

Validates auto-tune, the schedule, the file format and the loader with mock instruments and a
compressed protocol. Output goes to `co2dot_data/<timestamp>_mock_dryrun2`.

In [ ]:
# lic = MockLicor(tau_s={"co2": 2, "t": 2, "rh": 2})
# mdots = mock_dots2(2, lic)
# autotune(mdots, led_ma=4, target_counts=30000)
# dry = Run2(Path("co2dot_data") / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + "_mock_dryrun2"),
#            mdots, lic, Protocol2.quick(), led_ma=4)
# dry.run()
# cyc, ev, meta = load_run(dry.outdir); print(cyc.shape); print(ev["kind"].value_counts())

## 5. Flash timing check

With the LED at 4 mA the auto-tune pushes atime up, and one `spec_flash` (dark + lit integration)
gets slower. Run this after the auto-tune and before the loop: it warns if a flash takes more than
1 s on any device, or if one sample of all devices does not fit in the 5 s period.

In [ ]:
FLASH_WARN_S = 1.0
flash_s = {}
for dev_id, h in dots.items():
    t0 = time.perf_counter()
    for _ in range(3):
        h.dot.spec_flash(LED_MA)
    flash_s[dev_id] = (time.perf_counter() - t0) / 3
    st = h.dot.spec_status()
    flag = "  WARNING: flash > %.0f s (lower atime or raise the LED current)" % FLASH_WARN_S if flash_s[dev_id] > FLASH_WARN_S else ""
    print(f"[{h.label} {dev_id}] flash {flash_s[dev_id]:.2f} s  gain={st.get('gain')} atime={st.get('atime')} astep={st.get('astep')}{flag}")
per_sample = sum(flash_s.values()) + 0.3 * len(dots)          # + env read and overhead per device
print(f"one sample of all {len(dots)} device(s) ~{per_sample:.1f} s  (sample period {P.sample_period_s} s)")
if per_sample > 0.8 * P.sample_period_s:
    print("WARNING: sampling will fall behind the period: fewer devices, a longer sample_period_s, or a lower atime")

## 6. Run

Starts the scheduler (blocks this cell). Ctrl-C / interrupt the kernel stops cleanly: the chamber is
returned to ambient, the checkpoint is kept. Run the **resume** cell to continue. If the run
returns `'paused'`, the CO2 cartridge is probably empty: replace it and resume.

In [ ]:
RUN_TAG = "calib2"
run = Run2(Path("co2dot_data") / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + f"_{RUN_TAG}"),
           dots, licor, P, led_ma=LED_MA,
           meta_extra={"autotune": tune, "operator": "",
                       "notes": "devices in chamber facing the air stream; thermocouple on film A"})
print("run folder:", run.outdir)
status = run.run()
print("status:", status)

In [ ]:
# Resume after Ctrl-C / cartridge swap (same run object -> same folder and checkpoint):
# status = run.run(); print(status)

# After a kernel restart: re-run cells 1-3 first. Re-running autotune may give slightly different
# gain/atime; the settings of the original run are in meta.json under "spectrometer_settings".
# run = Run2(Path("co2dot_data") / "<folder>", dots, licor, P, led_ma=LED_MA); status = run.run()

## 7. Quick look

In [ ]:
import matplotlib.pyplot as plt
cyc, ev, meta = load_run(run.outdir)
print(cyc.shape, "samples x devices;", len(ev), "events;", cyc["phase_pass_"].nunique(), "pass(es)")
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for lab, g in cyc.groupby("label"):
    ax[0].plot(g["ts"], g["lr_630_515"], ".", ms=3, label=lab)
    ax[2].plot(g["ts"], g["env_T"], ".", ms=2, label=f"{lab} board T"); ax[2].plot(g["ts"], g["env_RH"], ".", ms=2, label=f"{lab} board RH")
g0 = cyc.drop_duplicates("cycle")
ax[1].plot(g0["ts"], g0["licor_CO2_s"], "k.", ms=2); ax[1].set_ylabel("CO2_s (ppm)")
ax[2].plot(g0["ts"], g0["licor_Tchamber"], "k.", ms=2, label="Tchamber"); ax[2].plot(g0["ts"], g0["licor_RHcham"], "k.", ms=2, label="RHcham")
ax[0].set_ylabel("log10(630/515)"); ax[0].legend(ncol=4, fontsize=8); ax[2].legend(ncol=4, fontsize=8)
plt.tight_layout()

# brightest lit channel per window: should stay inside 50-60 % of full scale (32 800-39 300 counts) in the brightest state
lit_cols = [c for c in cyc.columns if c.startswith("lit_")]
cyc["lit_peak"] = cyc[lit_cols].max(axis=1)
peak = cyc.groupby(["label", "phase_co2_nominal"])["lit_peak"].agg(["mean", "max"]).round(0)
print(peak, "\n")
sp = settled_points(cyc, last_s=60)
sp[["phase_step", "label", "pass_", "t_set", "rh_set", "co2_set", "n", "licor_CO2_s", "env_T", "env_RH", "lr_630_515"]].round(3)

## 8. Close

In [ ]:
run.close()        # closes every CO2Dot port
li.close()